# Best cases for all Ye & Ghassemi fracture samples

This notebook creates an AGU/JGR: Solid Earth-style version of the Ye & Ghassemi Figure-7 layout for SW-T1, SW-T2, SW-S3, and SW-S4. Each sample retains three stacked hydromechanical plots and the Stage 1–3 annotations.

Solid colored curves are the selected numerical result. The digitized Ye & Ghassemi (2018) validation histories are shown only as open points, using the same color as their matching numerical curve. The selected case names are entered manually below.

In [ ]:
%matplotlib inline

from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_project_root(start=Path.cwd()):
    """Locate the ORCA project whether Jupyter starts here or in a parent folder."""
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'plot_best_cases_figure7.py').is_file():
            return candidate
    raise FileNotFoundError('Could not locate scripts/plot_best_cases_figure7.py')


PROJECT_ROOT = find_project_root()
SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'plot_best_cases_figure7.py'

spec = importlib.util.spec_from_file_location('best_cases_figure7', SCRIPT_PATH)
plotting = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = plotting
spec.loader.exec_module(plotting)

print(f'Project root: {PROJECT_ROOT}')

## Configuration

Enter ranking-case IDs here (without result-directory suffixes such as `_hpc`). These defaults are the four final BBFast calibrations selected by the balanced five-channel Table 2 comparison.

In [ ]:
SELECTED_CASES = {
    'SWT1': '107_01_swt1_coh27p2_apscale0p01512_ppfix',
    'SWT2': '100_04_swt2_apscale0p0177_ppfix',
    'SWS3': '100_06_sw3_resc1p30_unld0p00_ppfix',
    'SWS4': '93_07_sw4_final_theta30_jrc5_ppfix',
}

OUTPUT_DIR = PROJECT_ROOT / 'doc' / 'ye_ghassemi_valdiation' / 'figures'
DPI = 400

## Resolve and verify the selected cases

This cell requires every manually entered case to occur exactly once in the reproducible ranking CSV. It then shows the selected file, updated rank, balanced error, and selection status before plotting.

In [ ]:
ranking = pd.read_csv(plotting.RANKING_PATH)
selected = plotting.select_best_cases(ranking, SELECTED_CASES)
selection_table = pd.DataFrame([
    {
        'sample': sample,
        'case': selected[sample]['case'],
        'rank': int(selected[sample]['rank_within_sample']),
        'mean_nRMSE_pct': float(selected[sample]['mean_nrmse_pct']),
        'selection_status': selected[sample]['selection_status'],
    }
    for sample in plotting.SAMPLE_ORDER
])
display(selection_table.style.format({'mean_nRMSE_pct': '{:.3f}'}))

## Figure-7 layout with validation overlays

Every available validation channel is drawn on the corresponding colored axis. Piston and production-pressure validation points appear only for samples where those digitized histories are available.

In [ ]:
figure, selected = plotting.build_figure(ranking, SELECTED_CASES)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pdf_path = OUTPUT_DIR / 'YE2018_BEST_CASES_HYDROMECHANICAL.pdf'
png_path = OUTPUT_DIR / 'YE2018_BEST_CASES_HYDROMECHANICAL.png'
figure.savefig(pdf_path, bbox_inches='tight')
figure.savefig(png_path, dpi=DPI, bbox_inches='tight')
plt.show()

## Saved publication files

In [ ]:
print(f'PDF: {pdf_path}')
print(f'PNG: {png_path}')

## Table 2 hold-stage comparisons

The same manually selected cases are evaluated at the 11 loading/unloading holds reported in Ye & Ghassemi (2018), Table 2. Numerical results are solid curves; Table 2 observations are same-colored open points without connecting lines. The mechanical and hydraulic quantities are separated into two AGU-width pages for legibility. Numerical normal dilation and shear slip use the first hold as their zero datum, consistent with the comparison/ranking workflow.

In [ ]:
TABLE2_SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'plot_best_cases_table2_agu.py'
table2_spec = importlib.util.spec_from_file_location(
    'best_cases_table2_agu', TABLE2_SCRIPT_PATH
)
table2_plotting = importlib.util.module_from_spec(table2_spec)
sys.modules[table2_spec.name] = table2_plotting
table2_spec.loader.exec_module(table2_plotting)

In [ ]:
table2_figures, table2_selection, table2_comparison = table2_plotting.build_figures(
    SELECTED_CASES
)
table2_paths = table2_plotting.save_outputs(
    table2_figures, table2_comparison, OUTPUT_DIR, dpi=DPI
)
display(table2_selection.style.format({'mean_nRMSE_pct': '{:.3f}'}))
plt.show()

In [ ]:
for output_name, output_path in table2_paths.items():
    print(f'{output_name}: {output_path}')

display(table2_comparison.head(12))

### Reporting note

The final selections are `107_01` for SW-T1, `100_04` for SW-T2, `100_06` for SW-S3, and `93_07` for SW-S4. SW-T1 reads its result from `results_csv_local`; the other three read from `results_csv_hpc_rorqual`. The ranking manifest records the exact source path, so no directory suffix belongs in `SELECTED_CASES`.